# 🧠 ESD RAG Support Assistant — Lalo @ Oportun

This notebook builds a **RAG (Retrieval-Augmented Generation)** system from your ServiceNow knowledge articles.

**What it does:**
1. Loads your SNOW articles from markdown files
2. Creates semantic embeddings using sentence-transformers
3. Indexes them with FAISS for fast search
4. When you query it, finds the most relevant articles and generates:
   - ✅ Troubleshooting steps
   - ✅ Ready-to-paste SNOW work notes
   - ✅ Slack messages to the user
   - ✅ Escalation path suggestions

---
## 📦 Step 1: Install Dependencies
Run this cell first — only needed once per session.

In [ ]:
!pip install -q sentence-transformers faiss-cpu

## 📁 Step 2: Upload Your Articles

Run this cell. It will open a file picker. Select all your `.md` article files.

**First time?** Start with 2-3 articles to test, then add more later.

In [ ]:
import os
from google.colab import files

# Create articles directory
os.makedirs('articles', exist_ok=True)

# Upload files
print("📂 Select your .md article files...")
uploaded = files.upload()

# Save uploaded files
for filename, content in uploaded.items():
    filepath = os.path.join('articles', filename)
    with open(filepath, 'wb') as f:
        f.write(content)
    print(f"  ✅ Saved: {filename}")

print(f"\n📚 Total articles in folder: {len(os.listdir('articles'))}")

## 🔧 Step 3: Build the RAG Engine

This cell:
1. Reads all your articles
2. Parses the metadata (system, category, escalation team)
3. Creates embeddings
4. Builds the FAISS index

Run it once after uploading articles. Re-run if you add new articles.

In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# --- Parse articles ---
def parse_article(filepath):
    """Parse a markdown article into structured data."""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    # Extract frontmatter metadata
    metadata = {}
    frontmatter_match = re.search(r'^---\s*\n(.+?)\n---', content, re.DOTALL)
    if frontmatter_match:
        for line in frontmatter_match.group(1).strip().split('\n'):
            if ':' in line:
                key, val = line.split(':', 1)
                metadata[key.strip()] = val.strip()

    # Extract sections
    sections = {}
    current_section = 'intro'
    sections[current_section] = []
    body = content
    if frontmatter_match:
        body = content[frontmatter_match.end():]

    for line in body.split('\n'):
        header_match = re.match(r'^##\s+(.+)', line)
        if header_match:
            current_section = header_match.group(1).strip().lower()
            sections[current_section] = []
        else:
            sections[current_section].append(line)

    # Clean sections
    for key in sections:
        sections[key] = '\n'.join(sections[key]).strip()

    return {
        'filepath': filepath,
        'filename': os.path.basename(filepath),
        'metadata': metadata,
        'sections': sections,
        'full_text': content
    }

# --- Load all articles ---
articles_dir = 'articles'
articles = []
for fname in sorted(os.listdir(articles_dir)):
    if fname.endswith('.md') and fname != 'TEMPLATE.md':
        filepath = os.path.join(articles_dir, fname)
        article = parse_article(filepath)
        articles.append(article)
        system = article['metadata'].get('system', '?')
        title = article['metadata'].get('title', fname)
        print(f"  📖 [{system}] {title}")

print(f"\n✅ Loaded {len(articles)} articles")

# --- Create embeddings ---
print("\n🧠 Loading embedding model (first run downloads ~90MB)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create search texts — combine key fields for better matching
search_texts = []
for art in articles:
    parts = [
        art['metadata'].get('title', ''),
        art['metadata'].get('system', ''),
        art['metadata'].get('category', ''),
        art['sections'].get('problem / symptom', ''),
        art['sections'].get('troubleshooting steps', ''),
        art['sections'].get('resolution', ''),
        art['sections'].get('notes', '')
    ]
    search_texts.append(' | '.join([p for p in parts if p]))

print("🔢 Creating embeddings...")
embeddings = model.encode(search_texts, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

# --- Build FAISS index ---
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"\n✅ FAISS index ready — {index.ntotal} articles indexed")
print("\n🚀 RAG engine is ready! Run the next cell to start querying.")

## 💬 Step 4: Query the RAG

Type a description of the issue (like a user would report it) and get back:
- Troubleshooting steps
- Ready-to-paste work notes for SNOW
- Slack message to send the user
- Escalation path if needed

**Run this cell as many times as you want!**

In [ ]:
def search_articles(query, top_k=3):
    """Search for the most relevant articles."""
    query_embedding = model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(articles):
            results.append({
                'article': articles[idx],
                'distance': distances[0][i],
                'relevance': max(0, 1 - distances[0][i] / 2)  # normalize 0-1
            })
    return results


def generate_work_notes(query, article):
    """Generate SNOW work notes from the matched article."""
    meta = article['metadata']
    sections = article['sections']
    steps = sections.get('troubleshooting steps', 'No steps documented.')
    resolution = sections.get('resolution', 'See troubleshooting steps.')

    note = f"""--- Work Notes ---
Issue reported: {query}
Matched KB: {meta.get('title', 'N/A')} ({meta.get('article_id', 'N/A')})
System: {meta.get('system', 'N/A')}

Troubleshooting performed:
{steps}

Resolution: {resolution}"""
    return note


def generate_slack_message(query, article, caller_name="[User Name]"):
    """Generate a Slack first-contact message."""
    meta = article['metadata']
    sections = article['sections']
    steps = sections.get('troubleshooting steps', '')

    # Extract first 3 steps for initial message
    step_lines = [l.strip() for l in steps.split('\n') if re.match(r'^\d+\.', l.strip())]
    first_steps = '\n'.join(step_lines[:3]) if step_lines else 'I will review your case.'

    msg = f"""Hello {caller_name}! :pika-wave:

My name is Lalo from IT Support, and I'm reaching out regarding your recent ticket.

I see you're experiencing an issue with *{meta.get('system', 'your application')}*. Let's try a few things:

{first_steps}

Could you try these steps and let me know the result? If the issue persists, I'll take a closer look.

You can also reach me on Zoom: https://oportun.zoom.us/my/lalo.esd

Thank you! :star2:"""
    return msg


def generate_escalation(article):
    """Generate escalation notes."""
    meta = article['metadata']
    sections = article['sections']
    esc = sections.get('escalation', 'No escalation info documented.')

    note = f"""⚠️ ESCALATION RECOMMENDATION
System: {meta.get('system', 'N/A')}
Escalation Team: {meta.get('escalation_team', 'N/A')}

{esc}"""
    return note


# --- Interactive query ---
print("=" * 60)
print("🔍 ESD RAG Support Assistant")
print("=" * 60)
query = input("\n📝 Describe the issue (or paste the Short Description):\n> ")

if query.strip():
    caller = input("\n👤 Caller name (or press Enter to skip):\n> ").strip() or "[User Name]"

    results = search_articles(query)

    if results:
        best = results[0]
        art = best['article']
        relevance_pct = f"{best['relevance']*100:.0f}%"

        print(f"\n{'=' * 60}")
        print(f"🎯 Best Match: {art['metadata'].get('title', art['filename'])}")
        print(f"   Relevance: {relevance_pct} | System: {art['metadata'].get('system', '?')}")
        print(f"{'=' * 60}")

        # 1. Troubleshooting steps
        print(f"\n\n{'─' * 40}")
        print("✅ TROUBLESHOOTING STEPS")
        print(f"{'─' * 40}")
        print(art['sections'].get('troubleshooting steps', 'No steps documented.'))

        # 2. Work notes
        print(f"\n\n{'─' * 40}")
        print("📝 SNOW WORK NOTES (copy-paste)")
        print(f"{'─' * 40}")
        print(generate_work_notes(query, art))

        # 3. Slack message
        print(f"\n\n{'─' * 40}")
        print("💬 SLACK MESSAGE (copy-paste)")
        print(f"{'─' * 40}")
        print(generate_slack_message(query, art, caller))

        # 4. Escalation
        print(f"\n\n{'─' * 40}")
        print("⚠️ ESCALATION PATH")
        print(f"{'─' * 40}")
        print(generate_escalation(art))

        # Other matches
        if len(results) > 1:
            print(f"\n\n{'─' * 40}")
            print("📚 OTHER POSSIBLE MATCHES")
            print(f"{'─' * 40}")
            for r in results[1:]:
                a = r['article']
                rel = f"{r['relevance']*100:.0f}%"
                print(f"  • [{rel}] {a['metadata'].get('title', a['filename'])} ({a['metadata'].get('system', '?')})")
    else:
        print("\n❌ No matching articles found. Try different keywords.")
else:
    print("No query entered.")

## 🔁 Run Again

Just re-run the cell above (Step 4) to query with a new issue.

To add more articles, re-run Step 2 to upload, then re-run Step 3 to rebuild the index.

---

### 💡 Tips
- **Query like a user would:** "I can't log into Zeus" works better than "Zeus login"
- **More articles = better results:** Start with your top 20-30 most common issues
- **Consistent format:** Use the template for every article — the metadata fields matter for escalation routing
- **Save to Drive:** File > Save a copy in Drive — so you don't lose your notebook